# Phase 4a: Contrastive Pair Construction (Candidate Generation)

Αυτό το notebook αναλαμβάνει το πρώτο μισό του Phase 4. Θα διαβάσει τα 318 missing features που φιλτράραμε και θα χρησιμοποιήσει το **Llama-3.1-8B-Instruct** για να δημιουργήσει (generate) υποψήφια τοξικά queries.

### ⚠️ ΣΗΜΑΝΤΙΚΟ: Hugging Face Token
Επειδή το τρέχεις πρώτη φορά στον λογαριασμό σου, πρέπει να κάνεις τα εξής:
1. Φτιάξε λογαριασμό στο [Hugging Face](https://huggingface.co/).
2. Πήγαινε στη σελίδα του [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) και πάτα αποδοχή των όρων χρήσης (συνήθως δίνουν έγκριση άμεσα).
3. Πήγαινε στα [Settings > Access Tokens](https://huggingface.co/settings/tokens) και φτιάξε ένα νέο Token (τύπου Read).
4. Στο μενού αριστερά στο Colab, πάτα το εικονίδιο με το κλειδί (Secrets), φτιάξε ένα νέο secret με όνομα **`HF_TOKEN`** και κάνε επικόλληση το token σου (επίλεξε το Notebook access ενεργό).

In [2]:
from google.colab import drive
drive.mount("/content/drive")

# Επιβεβαίωση ότι έχουμε GPU (T4 ή L4)
!nvidia-smi

Mounted at /content/drive
Mon May 25 21:32:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## 1. Εγκατάσταση Βιβλιοθηκών

In [ ]:
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.6 MB/s eta 0:00:00


## 2. Σύνδεση με Hugging Face

In [12]:
from google.colab import userdata
from huggingface_hub import login

# Θα τραβήξει αυτόματα το κλειδί που έβαλες στα Secrets του Colab
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

## 3. Λήψη Κώδικα (FAC-Synthesis)

In [13]:
%%bash
git clone https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git FAC-Synthesis

Cloning into 'FAC-Synthesis'...


## 4. Patch για 4-bit Llama Loading
Το Llama 3.1 8B κανονικά απαιτεί 16GB VRAM. Ανάλογα με τη GPU που σου έδωσε το Colab (π.χ. T4 έχει 15GB), μπορεί να краσάρει (Out Of Memory). Για να είμαστε 100% σίγουροι, πατσάρουμε το `llama_wrapper.py` για να το φορτώσει σε 4-bit (θέλει μόνο ~6GB VRAM), όπως κάνατε και στο Phase 2!

In [14]:
import os

wrapper_path = "/content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/llama_wrapper.py"

with open(wrapper_path, "r") as f:
    code = f.read()

patch = """from transformers import BitsAndBytesConfig
import torch
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)"""

code = code.replace("""model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)""", patch)

with open(wrapper_path, "w") as f:
    f.write(code)

print("✅ Το llama_wrapper.py ενημερώθηκε επιτυχώς για 4-bit precision!")

✅ Το llama_wrapper.py ενημερώθηκε επιτυχώς για 4-bit precision!


## 5. Αντιγραφή του TSV με τα Missing Features
Πρέπει να αντιγράψεις το αρχείο `intersection_tox7_corr3.tsv` (που έχει τα 318 features) από το Google Drive σου στο Colab.
*(Αν το έχεις σε διαφορετικό φάκελο στο Drive σου, άλλαξε το path παρακάτω)*

In [15]:
%%bash
# ΠΡΟΣΟΧΗ: Άλλαξε το path του MyDrive ανάλογα με το πού έχεις το αρχείο!
cp "/content/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/intersection_tox7_corr3___MISS_FEATURES__INPUT_FILE_STEP_4A_FIXED.tsv" /content/missing_features.tsv

# Επιβεβαίωση ότι ήρθε
ls -l /content/missing_features.tsv

-rw-r--r-- 1 root root 184315 May 25 16:31 /content/missing_features.tsv


## 6. Εκτέλεση του Generation (Phase 4a)
Τρέχουμε το script. Βάζουμε `--ratio 1.0` για να επεξεργαστεί **και τα 318 features** (χωρίς τυχαία δειγματοληψία) και `--num_synthetic_samples 2` για να παράγει 2 ερωτήματα ανά feature.

In [16]:
! cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python generate_data_llama_r1.py \
  --features /content/missing_features.tsv \
  --out "/content/drive/MyDrive/fac_synthesis/step_4/4a/log_files/step1_queries" \
  --max-features 5 \
  --ratio 1.00 \
  --num_synthetic_samples 2 \
  --temperature 0.8

tokenizer_config.json: 50.9kB [00:00, 67.2MB/s]
tokenizer.json: 9.09MB [00:00, 16.2MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 2.07MB/s]
config.json: 100% 855/855 [00:00<00:00, 4.57MB/s]
model.safetensors.index.json: 23.9kB [00:00, 90.3MB/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:00<?, ?B/s]
model-00001-of-00004.safetensors:   0% 0.00/4.98G [00:01<?, ?B/s]
model-00001-of-00004.safetensors:   1% 41.9M/4.98G [00:01<00:23, 209MB/s]
model-00001-of-00004.safetensors:   3% 131M/4.98G [00:02<00:54, 88.4MB/s]
model-00001-of-00004.safetensors:   7% 332M/4.98G [00:04<00:51, 90.1MB/s]
model-00001-of-00004.safetensors:  11% 533M/4.98G [00:04<00:26, 166MB/s] 
model-00001-of-00004.safetensors:  13% 667M/4.98G [0

## 7. Αντιγραφή Αποτελεσμάτων πίσω στο Drive
Μόλις τελειώσει, το παραγόμενο αρχείο θα λέγεται `step1_queries.queries.tsv`. Το στέλνουμε στο Drive για να μην χαθεί όταν κλείσει το Colab.

In [ ]:
%%bash
cp /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/step1_queries.queries.tsv /content/drive/MyDrive/step1_queries.queries.tsv
print("✅ Το αρχείο αποθηκεύτηκε στο Drive!")

# Phase 4b: SAE-Scoring & Contrastive Pair ConstructionΕδώ ξεκινάει το δεύτερο μισό του Phase 4 (Αξιολόγηση των κειμένων που παρήγαγε το Llama).Για να γίνει αυτό, χρειαζόμαστε τα βάρη του SAE (τον "εγκέφαλο"). Κατεβάζουμε το έτοιμο SAE checkpoint.

In [ ]:
from huggingface_hub import hf_hub_download
import os
import shutil

SAE_REPO = "Zhongzhi1228/sae_llama_l16_h65536"
SAE_FILENAME = "TopK7_l16_h4096_epoch3.pth"

print(f"Κατέβασμα SAE από το {SAE_REPO}...")
sae_cache_path = hf_hub_download(repo_id=SAE_REPO, filename=SAE_FILENAME)

# Αντιγραφή στον φάκελο που το περιμένει το script
os.makedirs("/content/sae_weights", exist_ok=True)
shutil.copy(sae_cache_path, "/content/sae_weights/topk_l16_h65536.pth")
print("✅ Το SAE κατέβηκε και τοποθετήθηκε σωστά μεσω huggingface_hub!")


## 8. Το "Κρυφό" Βήμα: Сollect Spans στα Συνθετικά ΔεδομέναΠερνάμε τις 636 προτάσεις που φτιάξαμε μέσα από το Llama+SAE για να πάρουμε τα activations τους.

In [ ]:
%%bash
cd /content/FAC-Synthesis/sae_feature_analysis/interpret_features/

python collect_spans.py 0 llama 0 1 \
    --data-path /content/drive/MyDrive/step1_queries.queries.tsv \
    --threshold 0.0 \
    --sae-path /content/sae_weights/topk_l16_h65536.pth \
    --out-dir /content/drive/MyDrive/out_synthetic


## 9. Τρέχουμε το Analyze
Θα διαβάσει τα σκορ από το σκανάρισμα και θα κρατήσει το Top-2 ανά feature.

In [ ]:
%%bash
cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/
python analyze_step1_synthetic_data.py \
  --final-decision-file /content/missing_features.tsv \
  --textspans-file /content/drive/MyDrive/out_synthetic/threshold_0.0/textspans_group0.tsv \
  --synthetic-queries-file /content/drive/MyDrive/step1_queries.queries.tsv \
  --output-jsonl /content/drive/MyDrive/step1_analyzed.jsonl


## 10. Τρέξιμο του merge_step1_failed_cases.py
Εδώ γίνεται η τελική δημιουργία των Contrastive Pairs (το αρχείο που πάει στο Round 2).

In [ ]:
!cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python merge_step1_failed_cases.py \
  --final-decision-file /content/missing_features.tsv \
  --triplets-file /content/drive/MyDrive/step1_analyzed.jsonl \
  --output-file /content/drive/MyDrive/step1_contrastive_pairs.jsonl


## 12. Αποθήκευση στο Google Drive
Στέλνουμε το χρυσό αρχείο (jsonl) στο Drive για να το χρησιμοποιήσουμε στο Phase 4c (Round 2).

In [ ]:
%%bash
cp /content/step1_contrastive_pairs.jsonl /content/drive/MyDrive/
print("✅ ΤΕΛΟΣ! Το step1_contrastive_pairs.jsonl είναι στο Drive σου.")